# Your selection, back in Python: `selectedDataFrame()` and `onSelection()`

Drag, pick or filter in an interactive layout, then read what you selected as a DataFrame
in the next cell (PLANNING.md §7 **F3**). The check that defines the feature as done:
**drag in a view below, then run the cell after it, and the rows are the ones you dragged.**

- **The selection is what the views show:** the whole frame until you drag, pick or filter,
  then that subset, and back as you pop the stack.
- **On `linkpi`, selected nodes narrow it** to the rows with a selected node at either end.
- **Brushing changes nothing here.** It is a preview that moves with the mouse.

Needs a live kernel and the `interactive` extra (`pip install polars2svg[interactive]`).

In [ ]:
import pathlib, sys
_root = pathlib.Path('..').resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import polars as pl
import panel as pn
from polars2svg import Polars2SVG
import polars2svg; print(polars2svg.__file__)
pn.extension()
p2s = Polars2SVG()

# A small netflow-shaped frame: a few busy servers, a tail of clients, one hour of traffic.
rng   = np.random.default_rng(20261003)
hosts = [f'10.0.{i // 20}.{i % 20 + 1}' for i in range(60)]
w     = 1.0 / np.arange(1, len(hosts) + 1) ** 0.9
w    /= w.sum()
n     = 4000
t0    = np.datetime64('2026-10-03T09:00:00', 'us')
df = pl.DataFrame({
    'ts':    pl.Series(t0 + rng.integers(0, 3_600_000_000, n).astype('timedelta64[us]')),
    'sip':   rng.choice(hosts, n, p=w),
    'dip':   rng.choice(hosts, n, p=w[::-1] / w.sum()),
    'dport': rng.choice([80, 443, 53, 22, 445, 3389], n, p=[.3, .35, .15, .08, .07, .05]),
    'bytes': np.exp(rng.normal(6.0, 2.0, n)).round(),
}).filter(pl.col('sip') != pl.col('dip'))
df.height

## 1. A layout: a scatter, a histogram and the graph, on one stack

In [ ]:
layout = p2s.panelize([[p2s.xyp(df, 'ts', 'bytes', color='dport', wxh=(640, 300)),
                        p2s.histop(df, 'dport', wxh=(256, 300))],
                       [p2s.linkp(df=df, relationships=[('sip', 'dip')], wxh=(640, 480))]])
layout

## 2. Read it back

Before you do anything this is the whole frame. Drag a box on the scatter (or a band on
the histogram), then run this cell again: the rows are the ones the views now show, with
your input columns.

In [ ]:
sel = layout.selectedDataFrame()
print(f'{sel.height} of {df.height} rows')
sel.head()

## 3. Be told when it changes

`onSelection(callback)` calls `callback(df)` after every change: a drag, a pop (drag over
empty space), a node selection on the graph. It also works as a decorator. Run this, make a
few selections, then run the cell after it.

In [ ]:
log = []

@layout.onSelection
def record(df):
    log.append((df.height, df['dport'].unique().sort().to_list()))

In [ ]:
log

## 4. Selected nodes narrow it

On the graph, drag a box around a few hosts (that selects them; it does not drill down).
The selection becomes the rows with a selected host at either end.

In [ ]:
sel = layout.selectedDataFrame()
print(f'{sel.height} of {df.height} rows (all of them until a host is selected)')
sel.group_by('sip', 'dip').len().sort('len', descending=True).head(10)

## 5. One view on its own

A view built outside `panelize()` has the same two methods, for its own stack.

In [ ]:
xi = p2s.xypi(p2s.xyp(df, 'ts', 'bytes', color='dport', wxh=(640, 300)))
xi

In [ ]:
xi.selectedDataFrame().height